In [1]:
import warnings
warnings.filterwarnings("ignore")

In [2]:
import os
import json

from dotenv import load_dotenv
load_dotenv()

import ollama

from langchain_google_genai import ChatGoogleGenerativeAI
from langchain.agents import create_agent
from langchain_core.messages import AIMessage
from langchain_core.tools import tool
from langgraph.prebuilt import InjectedState

from file_tools import (
    DeepAgentState,
    ls,
    read_file,
    write_file,
    cleanup_files,
)
from prompts import (
    ORCHESTRATOR_PROMPT,
    RESEARCHER_PROMPT,
    EDITOR_PROMPT,
)

In [3]:
# -------------------------
# Web Search Tool
# -------------------------

@tool
def web_search(query: str) -> str:
    """
    Perform a live web search using Ollama Cloud Web Search API.

    Input:
        query: search query string

    Output:
        JSON string of top results (max_results=2).
    """
    response = ollama.web_search(query, max_results=2)
    response = response.results

    return response


In [4]:
result = web_search.invoke({"query": "Latest global news"})

In [5]:
result

[WebSearchResult(content='US carries out &#x27;massive&#x27; strike against IS in Syria\n[Skip to content](#main-content)\n[Watch Live](https://www.bbc.com/watch-live-news/)\n[British Broadcasting Corporation](https://www.bbc.com/)\n[Home](https://www.bbc.com/)\nNews\n[Sport](https://www.bbc.com/sport)\nBusiness\nInnovation\nCulture\nArts\nTravel\nEarth\nAudio\nVideo\nLive\nDocumentaries\n[Weather](https://www.bbc.com/weather)\n[Newsletters](https://www.bbc.com/newsletters)\n[Watch Live](https://www.bbc.com/watch-live-news/)\n# US carries out &#x27;massive&#x27; strike against IS in Syria\n1 day ago\nShareSave\nJaroslav Lukiv\nShareSave\nUS President Donald Trump: &quot;We hit every site flawlessly&quot;\nThe US says its military has carried out a &quot;massive strike&quot; against the Islamic State group (IS) in Syria, in response to a deadly attack on American forces in the country.\nThe US Central Command (Centcom) said fighter jets, attack helicopters and artillery &quot;struck mor

In [6]:
# -------------------------
# LLM
# -------------------------

llm = ChatGoogleGenerativeAI(model="gemini-3-flash-preview")
# llm = ChatGoogleGenerativeAI(model="gemini-3-pro-preview")

In [7]:
# -------------------------
# Worker Agents: Researcher & Editor
# -------------------------
researcher_tools = [ls, write_file, read_file, web_search]

researcher_agent = create_agent(
    model=llm,
    tools=researcher_tools,
    system_prompt=RESEARCHER_PROMPT,
    state_schema=DeepAgentState,
)

editor_tools = [ls, read_file, write_file, cleanup_files]
editor_agent = create_agent(
    model=llm,
    tools=editor_tools,
    system_prompt=EDITOR_PROMPT,
    state_schema=DeepAgentState,
)


In [8]:
# -------------------------
# Orchestrator Tools (call other agents)
# -------------------------
from typing import Annotated
from langgraph.prebuilt import InjectedState
from langchain_core.tools import InjectedToolCallId
from langgraph.types import Command

@tool
def run_researcher(
    theme_id: int,
    thematic_question: str,
    state: Annotated[DeepAgentState, InjectedState],
    max_retries: int = 2
) -> str:
    """
    Run a single Research agent for ONE thematic question.

    Args:
        theme_id: The theme number (1, 2, 3, etc.)
        thematic_question: The specific thematic question to research
        state: Injected agent state providing user_id/thread_id
        max_retries: Number of retry attempts if researcher fails

    The Researcher will:
    - Receive ONE thematic question as input
    - Break it into 2-4 focused search queries
    - Use web_search to gather information
    - Write files to researcher/ folder with hash-based names:
        * researcher/<hash>_theme.md (research findings)
        * researcher/<hash>_sources.txt (raw sources)

    Returns a short status string for the orchestrator.
    """
    from file_tools import generate_hash
    
    # Generate hash for unique file naming
    file_hash = generate_hash(f"{theme_id}_{thematic_question}")
    
    # Build sub-state with theme-specific context
    sub_state: DeepAgentState = {
        "messages": state["messages"] + [
            AIMessage(content=f"[THEME {theme_id}] Research this question: {thematic_question}\n\n"
                             f"File hash: {file_hash}\n"
                             f"Save your findings to: researcher/{file_hash}_theme.md\n"
                             f"Save your sources to: researcher/{file_hash}_sources.txt")
        ],
        "user_id": state.get("user_id"),
        "thread_id": state.get("thread_id"),
    }
    
    # Execute researcher with retry logic
    for attempt in range(max_retries + 1):
        try:
            researcher_agent.invoke(sub_state)
            return f"✓ Theme {theme_id} research completed (hash: {file_hash})"
        except Exception as e:
            if attempt < max_retries:
                print(f"⚠ Theme {theme_id} failed (attempt {attempt + 1}/{max_retries + 1}), retrying...")
                continue
            else:
                return f"✗ Theme {theme_id} failed after {max_retries + 1} attempts: {str(e)}"
    
    return f"✓ Theme {theme_id} research completed (hash: {file_hash})"


In [ ]:
@tool
def write_research_plan(
    thematic_questions: list[str],
    state: Annotated[DeepAgentState, InjectedState],
    tool_call_id: Annotated[str, InjectedToolCallId],
) -> Command:
    """
    Write the high-level research plan with major thematic questions.

    Args:
        thematic_questions: List of 3-5 major thematic questions that break down
                           the user's query. These guide the Researcher agent.
        state: Injected agent state providing user_id/thread_id.
        tool_call_id: Tool call ID for attaching a ToolMessage.

    The research_plan.md will be read by the Researcher to guide tactical research.

    Returns:
        Command with a ToolMessage confirming the plan was written.
    """
    from file_tools import _disk_path
    from langchain_core.messages import ToolMessage
    from langgraph.types import Command
    
    # Build content for research_plan.md
    content = "# Research Plan\n\n"
    content += "## User Query\n"
    # Extract user query from messages
    user_msg = [m for m in state["messages"] if hasattr(m, 'content')]
    if user_msg:
        content += f"{user_msg[-1].content}\n\n"
    
    content += "## Thematic Questions\n\n"
    for i, question in enumerate(thematic_questions, 1):
        content += f"{i}. {question}\n"
    
    # Write to disk
    path = _disk_path(state, "research_plan.md")
    with open(path, "w", encoding="utf-8") as f:
        f.write(content)
    
    msg = f"[RESEARCH PLAN WRITTEN] research_plan.md with {len(thematic_questions)} thematic questions -> {path}"
    return Command(
        update={
            "messages": [ToolMessage(msg, tool_call_id=tool_call_id)]
        }
    )


@tool
def run_editor(
    state: Annotated[DeepAgentState, InjectedState]
) -> str:
    """
    Run the Editor agent for this user/thread.

    The Editor will:
    - read research_plan.md, theme files, sources.txt
    - write the final answer to report.md

    Returns a short status string for the orchestrator.
    """
    sub_state: DeepAgentState = {
        "messages": [],
        "user_id": state.get("user_id"),
        "thread_id": state.get("thread_id"),
    }
    response = editor_agent.invoke(sub_state)
    # return "Editor completed. Final report is written to report.md."
    return f"Ideally report.md should be generated by now but check it with following response if it is generate othwise thell the issues. Here is editor agent response. Response:\n\n{response}"

In [10]:
# -------------------------
# Orchestrator Agent
# -------------------------

orchestrator_tools = [
    write_research_plan,  # strategic planning
    run_researcher,       # tactical research
    run_editor,           # synthesis
    cleanup_files,        # resetting workspace
]

orchestrator_agent = create_agent(
    model=llm,
    tools=orchestrator_tools,
    system_prompt=ORCHESTRATOR_PROMPT,
    state_schema=DeepAgentState,
)

In [11]:
# orchestrator_agent
# editor_agent


### Example usage


In [12]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Tell me a joke about computers.")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)
result

{'messages': [HumanMessage(content='Tell me a joke about computers.', additional_kwargs={}, response_metadata={}, id='37295641-a84c-4629-baa4-dd4ea393e835'),
  AIMessage(content=[{'type': 'text', 'text': 'Why did the computer show up late to work?\n\nBecause it had a hard drive!', 'extras': {'signature': 'Ev0BCvoBAXLI2nzVvFhz26F/a4BUklD12Z1in17rUIne/QGpd2KXqv2WyFN+6juyt+dYnEk0e4xfVmGQJFXNOiZv/Sr1uB5XeQChn31kTeBvEnCfBMLoc1akPq7vaFKMe8xR3HnYbulavLWtm4KR0lWp19jpApxUGoNFNj7IPqC8NdFciyI35n2lXAX1p1HXLsKQ4fMQnq3GA20j6ZD9Ar9aEdKcg7Mk0mUTMzjqZcK9pXbemV2ajolFx28pOa4tVUd13TjT01MTOES3YDlDj19C9M1rNErkvIf73ULCyyD0gy+2q/zdmYifv5S/fEI9lnaoDW5OOQLo6qcmJdrryQ=='}}], additional_kwargs={}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--8db51e8f-7d25-4675-9ef5-aaf7ab6bdbd8-0', usage_metadata={'input_tokens': 2015, 'output_tokens': 57, 'total

In [13]:
from langchain.messages import HumanMessage
state = {
        "messages": [HumanMessage(content="Do a detailed, well-structured analysis of MCP including history")],
        "user_id": "user_123",
        "thread_id": "thread_mcp_001",
    }

result = orchestrator_agent.invoke(state)

In [14]:
result

{'messages': [HumanMessage(content='Do a detailed, well-structured analysis of MCP including history', additional_kwargs={}, response_metadata={}, id='e5b21a25-fcac-4fbc-8886-ad57472891c4'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'write_research_plan', 'arguments': '{"thematic_questions": ["What is the Model Context Protocol (MCP), who created it, and what problem does it aim to solve in the AI ecosystem?", "What is the history and evolution of MCP, including its announcement and the motivation behind its development?", "What are the core architectural elements of MCP (e.g., hosts, servers, clients) and how do they interact?", "What are the primary applications, practical use cases, and current integrations of MCP?", "What are the advantages, limitations, and potential future impact of MCP on AI connectivity?"]}'}, '__gemini_function_call_thought_signatures__': {'3dc5871c-2a4f-41c8-a673-3471c2b3464a': 'EpsLCpgLAXLI2nyQx1ycIiGcPuYng7tQtye1gg+kswIC/wNeS6vISb

In [17]:
from langchain.messages import HumanMessage

state = {
    "messages": [HumanMessage(content="Generate the final report")],
    "user_id": "user_123",
    "thread_id": "thread_mcp_001",
}

# This will now work because ls() can list the researcher folder

result = editor_agent.invoke(state)
result

{'messages': [HumanMessage(content='Generate the final report', additional_kwargs={}, response_metadata={}, id='8b4be5e9-f157-434e-bca8-caec403d9259'),
  AIMessage(content=[], additional_kwargs={'function_call': {'name': 'ls', 'arguments': '{"path": ""}'}, '__gemini_function_call_thought_signatures__': {'9f02605d-1851-4879-9e01-65dbe3716f89': 'EvQBCvEBAXLI2ny0JLV34I22n/gU3jYV2pilELsv7oNnIJPxKvMnzko7eyCnX5WTCiKMikfX63xv0DLEArHotghd/KpverJt/JZB48i/6OXZ4XnXu8gm0GG65tvyohZ8XjFs18wl+50sveekspzXqtvMcWP8XJC7MVEJSoxqjbKgfQiqwa4bgESikeRzrMy3hS7+EpzXFmBmDBFoSTTnGCokxE9AMmMmCVjKbZOLRW1a+tlHQf1UXjsnTAMvfrMzpGo11/3CtQDaxgpSIWvEmX6L5SHAL6Uj7tfpxFFS7vyXusRgKfgQHMewbZ4WFUyOii/qRFLjDQ=='}}, response_metadata={'prompt_feedback': {'block_reason': 0, 'safety_ratings': []}, 'finish_reason': 'STOP', 'model_name': 'gemini-3-flash-preview', 'safety_ratings': [], 'model_provider': 'google_genai'}, id='lc_run--bd684513-0346-4920-8322-57658ec158d2-0', tool_calls=[{'name': 'ls', 'args': {'path': ''}, 'id': '9f026